# Cross-dataset statistical analysis and paper plots

Friedman/Nemenyi comparisons, runtime, forecaster $R^2$, public per-horizon validity, and dataset-dependence checks. 

`scikit-posthocs` is installed through `requirements.txt`.


In [ ]:
# If scikit-posthocs is not installed in this environment, run:
# %pip install scikit-posthocs

from pathlib import Path
import math
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, studentized_range

try:
    import scikit_posthocs as sp
except ImportError as exc:
    raise ImportError(
        "scikit-posthocs is required. Install it with: pip install scikit-posthocs"
    ) from exc


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    """Locate the repository root from the current working directory."""
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "Data").is_dir() and (candidate / "Results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate CFE_for_MTS_Forecasting_Framework. "
        "Run this notebook from somewhere inside the cloned repository."
    )

PROJECT_ROOT = find_project_root()
print("Project root:", PROJECT_ROOT)

BASE_DIR = PROJECT_ROOT
BRGM_SUMMARY_PATH = PROJECT_ROOT / "Results" / "EDBT-results" / "counterfactual_summary_by_dataset_method.csv"
PUBLIC_SUMMARY_PATH = PROJECT_ROOT / "Results" / "EDBT-results-public-datasets" / "counterfactual_summary_by_dataset_method.csv"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "nemenyi_results_corrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:")
print(BASE_DIR)

print("\nBRGM summary:")
print(BRGM_SUMMARY_PATH)
print("Exists?", BRGM_SUMMARY_PATH.exists())

print("\nPublic summary:")
print(PUBLIC_SUMMARY_PATH)
print("Exists?", PUBLIC_SUMMARY_PATH.exists())

print("\nOutput directory:")
print(OUTPUT_DIR)


In [ ]:
# ============================================================
# METHODS AND METRIC DIRECTIONS
# ============================================================

METHOD_ORDER = [
    "ForecastCF-target",
    "ForecastCF-exog",
    "COMET-exog",
    "DiffusionCF-target",
    "DiffusionCF-exog",
    "NTD-CFE-exog",
    "GA-exog",
]

METRICS = {
    "validity_ratio": {
        "label": "Validity ratio",
        "higher_is_better": True,
    },
    "stepwise_validity_auc": {
        "label": "Prefix validity",
        "higher_is_better": True,
    },
    "consecutive_valid_steps": {
        "label": "Prefix valid steps",
        "higher_is_better": True,
    },
    "max_consecutive_valid_steps": {
        "label": "Maximum consecutive valid steps",
        "higher_is_better": True,
    },
    "distance_to_band": {
        "label": "Distance to band",
        "higher_is_better": False,
    },
    "proximity_l2_scaled": {
        "label": "Scaled L2 proximity",
        "higher_is_better": False,
    },
    "proximity_l1_scaled": {
        "label": "Scaled L1 proximity",
        "higher_is_better": False,
    },
    "sparsity_ratio": {
        "label": "Sparsity ratio",
        "higher_is_better": False,
    },
    "runtime_sec": {
        "label": "Runtime",
        "higher_is_better": False,
    },
}

ALPHA = 0.05
EXPECTED_DATASETS = 21
EXPECTED_METHODS = 7


In [ ]:
# ============================================================
# LOAD AND VALIDATE THE SAVED EXPERIMENT RESULTS
# ============================================================

brgm = pd.read_csv(BRGM_SUMMARY_PATH)
public = pd.read_csv(PUBLIC_SUMMARY_PATH)

print("BRGM summary shape:", brgm.shape)
print("Public summary shape:", public.shape)

required_columns = {"dataset", "method", *METRICS.keys()}

for source_name, df in [("BRGM", brgm), ("Public", public)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(
            f"{source_name} summary is missing required columns: {sorted(missing)}"
        )

results = pd.concat([brgm, public], ignore_index=True)

duplicates = results.groupby(["dataset", "method"]).size()
duplicates = duplicates[duplicates > 1]

if len(duplicates):
    raise ValueError(
        "Duplicate dataset-method rows were found. "
        "Use counterfactual_summary_by_dataset_method.csv, not raw-window results.\n\n"
        + duplicates.to_string()
    )

datasets = sorted(results["dataset"].astype(str).unique())
methods_found = set(results["method"].astype(str).unique())

if len(datasets) != EXPECTED_DATASETS:
    raise ValueError(
        f"Expected {EXPECTED_DATASETS} datasets, found {len(datasets)}."
    )

missing_methods = set(METHOD_ORDER) - methods_found
extra_methods = methods_found - set(METHOD_ORDER)

if missing_methods:
    raise ValueError(f"Missing expected methods: {sorted(missing_methods)}")
if extra_methods:
    raise ValueError(f"Unexpected methods found: {sorted(extra_methods)}")

coverage = results.groupby("dataset")["method"].nunique()
if not (coverage == EXPECTED_METHODS).all():
    raise ValueError(
        "Every dataset must contain all seven methods.\n\n"
        + coverage.to_string()
    )

if len(results) != EXPECTED_DATASETS * EXPECTED_METHODS:
    raise ValueError(
        f"Expected exactly {EXPECTED_DATASETS * EXPECTED_METHODS} rows, "
        f"found {len(results)}."
    )

results.to_csv(
    OUTPUT_DIR / "combined_21_dataset_method_summary.csv",
    index=False,
)

print("\nIntegrity checks passed.")
print(f"Datasets: {len(datasets)}")
print(f"Methods:  {len(METHOD_ORDER)}")
print(f"Rows:     {len(results)}")


In [ ]:
# ============================================================
# HELPERS: VALUE TABLES, RANKS, CD
# ============================================================

def build_value_and_rank_tables(metric, higher_is_better):
    """
    Return the 21x7 value table and within-dataset rank table.
    Rank 1 is always best.
    """
    value_table = (
        results.pivot(
            index="dataset",
            columns="method",
            values=metric,
        )
        .reindex(columns=METHOD_ORDER)
        .apply(pd.to_numeric, errors="coerce")
    )

    if value_table.isna().any().any():
        bad = value_table[value_table.isna().any(axis=1)]
        raise ValueError(
            f"Missing/non-numeric values for metric '{metric}':\n{bad}"
        )

    rank_table = value_table.rank(
        axis=1,
        method="average",
        ascending=not higher_is_better,
    )

    return value_table, rank_table


def dem_sar_nemenyi_cd(k, n_datasets, alpha=0.05):
    """
    Standard Demsar Nemenyi critical difference:
        CD = q_alpha * sqrt(k(k+1)/(6N))
    """
    q_alpha = (
        studentized_range.ppf(
            1.0 - alpha,
            k,
            np.inf,
        )
        / math.sqrt(2.0)
    )

    cd = q_alpha * math.sqrt(
        k * (k + 1) / (6.0 * n_datasets)
    )

    return float(cd), float(q_alpha)


def check_rank_direction(value_table, rank_table, higher_is_better, metric):
    """
    Verify that the best raw value(s) in each dataset receive
    the minimum rank for that dataset.
    """
    for dataset in value_table.index:
        row = value_table.loc[dataset]
        ranks = rank_table.loc[dataset]

        best_value = row.max() if higher_is_better else row.min()
        best_methods = row.index[np.isclose(row.values, best_value)]
        minimum_rank = ranks.min()

        if not np.allclose(
            ranks.loc[best_methods].values,
            minimum_rank,
        ):
            raise AssertionError(
                f"Ranking direction failed for {metric} on dataset {dataset}."
            )


In [ ]:
# ============================================================
# HELPERS: NON-SIGNIFICANT GROUPS FROM NEMENYI P-VALUES
# ============================================================

def maximal_nonsignificant_cliques(
    average_ranks,
    p_matrix,
    alpha=0.05,
):
    """
    Create the graph of pairwise non-significant Nemenyi comparisons
    (p >= alpha) and return maximal cliques of size >= 2.

    This uses the actual Nemenyi p-value matrix from scikit-posthocs.
    """
    methods = list(average_ranks.sort_values().index)

    def is_clique(subset):
        for a, b in itertools.combinations(subset, 2):
            if float(p_matrix.loc[a, b]) < alpha:
                return False
        return True

    cliques = []

    for size in range(2, len(methods) + 1):
        for subset in itertools.combinations(methods, size):
            if is_clique(subset):
                cliques.append(set(subset))

    maximal = []
    for clique in cliques:
        if not any(clique < other for other in cliques):
            maximal.append(clique)

    unique = []
    seen = set()

    for clique in maximal:
        key = tuple(sorted(clique))
        if key not in seen:
            seen.add(key)
            unique.append(clique)

    ordered_ranks = average_ranks.sort_values()
    positions = {
        m: float(ordered_ranks[m])
        for m in ordered_ranks.index
    }

    groups = []

    for clique in unique:
        ordered_methods = sorted(
            clique,
            key=lambda m: positions[m],
        )

        groups.append({
            "methods": ordered_methods,
            "left": positions[ordered_methods[0]],
            "right": positions[ordered_methods[-1]],
        })

    groups.sort(
        key=lambda g: (g["left"], g["right"])
    )

    return groups


def assign_bar_levels(groups):
    """
    Place overlapping crossbars on different vertical levels.
    """
    assigned = []
    levels = []

    for group in groups:
        left = group["left"]
        right = group["right"]
        level = 0

        while True:
            conflict = any(
                old_level == level
                and not (
                    right < old_left
                    or old_right < left
                )
                for old_left, old_right, old_level in assigned
            )

            if not conflict:
                break

            level += 1

        assigned.append((left, right, level))
        levels.append(level)

    return levels


In [ ]:
# ============================================================
# BLACK PUBLICATION-STYLE CD DIAGRAM
# ============================================================

def plot_cd_diagram(
    average_ranks,
    p_matrix,
    cd,
    metric_label,
    out_path,
    alpha=0.05,
    n_datasets=21,
):
    """
    Black publication-style critical-difference diagram.

    Statistical crossbars come from the scikit-posthocs Nemenyi
    p-value matrix. A crossbar means no significant pairwise
    difference was detected among all methods in that maximal clique.
    """
    ordered = average_ranks.sort_values()
    methods = ordered.index.tolist()
    ranks = ordered.to_numpy(dtype=float)
    k = len(methods)

    groups = maximal_nonsignificant_cliques(
        average_ranks=ordered,
        p_matrix=p_matrix,
        alpha=alpha,
    )

    group_levels = assign_bar_levels(groups)

    n_left = (k + 1) // 2
    left_idx = list(range(n_left))
    right_idx = list(range(n_left, k))

    fig, ax = plt.subplots(figsize=(12.8, 5.4))
    axis_y = 0.0

    ax.hlines(
        axis_y,
        1,
        k,
        color="black",
        linewidth=1.3,
    )

    for r in np.arange(1.0, k + 0.001, 0.5):
        major = abs(r - round(r)) < 1e-10
        half_height = 0.075 if major else 0.04

        ax.vlines(
            r,
            axis_y - half_height,
            axis_y + half_height,
            color="black",
            linewidth=1.2 if major else 0.9,
        )

    for r in range(1, k + 1):
        ax.text(
            r,
            0.17,
            str(r),
            ha="center",
            va="bottom",
            fontsize=11,
            color="black",
        )

    group_y0 = 0.38
    group_gap = 0.13

    for group, lev in zip(groups, group_levels):
        y = group_y0 + lev * group_gap

        ax.plot(
            [group["left"], group["right"]],
            [y, y],
            color="black",
            linewidth=4.0,
            solid_capstyle="butt",
        )

    max_level = max(group_levels, default=-1)

    cd_y = max(
        0.78,
        group_y0
        + (max_level + 1) * group_gap
        + 0.18,
    )

    cd_start = 1.0
    cd_end = min(k, cd_start + cd)

    ax.plot(
        [cd_start, cd_end],
        [cd_y, cd_y],
        color="black",
        linewidth=2.2,
    )

    ax.vlines(
        [cd_start, cd_end],
        cd_y - 0.07,
        cd_y + 0.07,
        color="black",
        linewidth=1.8,
    )

    ax.text(
        cd_start,
        cd_y + 0.13,
        f"CD = {cd:.3f}",
        ha="left",
        va="bottom",
        fontsize=10.5,
        color="black",
    )

    left_x = 0.72
    right_x = k + 0.28

    left_ys = np.linspace(
        -0.30,
        -1.28,
        len(left_idx),
    )

    right_ys = np.linspace(
        -0.30,
        -1.05,
        max(1, len(right_idx)),
    )

    for idx, y in zip(left_idx, left_ys):
        method = methods[idx]
        rank = ranks[idx]

        ax.plot(
            [rank, rank],
            [axis_y, y],
            color="black",
            linewidth=0.9,
        )

        ax.plot(
            [left_x, rank],
            [y, y],
            color="black",
            linewidth=0.9,
        )

        ax.text(
            left_x - 0.06,
            y,
            method,
            ha="right",
            va="center",
            fontsize=10.5,
            color="black",
        )

    for idx, y in zip(right_idx, right_ys):
        method = methods[idx]
        rank = ranks[idx]

        ax.plot(
            [rank, rank],
            [axis_y, y],
            color="black",
            linewidth=0.9,
        )

        ax.plot(
            [rank, right_x],
            [y, y],
            color="black",
            linewidth=0.9,
        )

        ax.text(
            right_x + 0.06,
            y,
            method,
            ha="left",
            va="center",
            fontsize=10.5,
            color="black",
        )

    ax.text(
        (1 + k) / 2,
        cd_y + 0.31,
        f"Critical difference diagram: {metric_label}",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold",
        color="black",
    )

    ax.text(
        (1 + k) / 2,
        -1.62,
        (
            f"Lower rank is better. N={n_datasets} datasets, "
            f"k={k} methods; Friedman omnibus followed by "
            f"Nemenyi post-hoc, alpha={alpha:.2f}. "
            "Thick bars indicate no significant pairwise "
            "difference detected."
        ),
        ha="center",
        va="top",
        fontsize=9.3,
        color="black",
    )

    ax.set_xlim(0.48, k + 0.52)
    ax.set_ylim(-1.87, cd_y + 0.52)

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()

    out_path = Path(out_path)

    fig.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        out_path.with_suffix(".pdf"),
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


In [ ]:
# ============================================================
# RUN FRIEDMAN + SCIKIT-POSTHOCS NEMENYI FOR EVERY METRIC
# ============================================================

friedman_rows = []
all_rank_rows = []

for metric, info in METRICS.items():

    print("\n" + "=" * 90)
    print(info["label"])
    print("=" * 90)

    value_table, rank_table = build_value_and_rank_tables(
        metric=metric,
        higher_is_better=info["higher_is_better"],
    )

    check_rank_direction(
        value_table=value_table,
        rank_table=rank_table,
        higher_is_better=info["higher_is_better"],
        metric=metric,
    )

    n_datasets = len(value_table)
    k = len(METHOD_ORDER)

    value_table.to_csv(
        OUTPUT_DIR
        / f"values_{metric}_21_datasets.csv"
    )

    rank_table.to_csv(
        OUTPUT_DIR
        / f"ranks_{metric}_by_dataset.csv"
    )

    # Friedman omnibus test.
    friedman_stat, friedman_p = friedmanchisquare(
        *[
            value_table[m].to_numpy(dtype=float)
            for m in METHOD_ORDER
        ]
    )

    # Average ranks used in the CD diagram.
    average_ranks = (
        rank_table
        .mean(axis=0)
        .sort_values()
    )

    average_ranks.rename(
        "average_rank"
    ).to_csv(
        OUTPUT_DIR
        / f"average_ranks_{metric}.csv"
    )

    # Authentic Nemenyi post-hoc test from scikit-posthocs.
    # Rows are datasets (blocks), columns are methods.
    nemenyi_result = sp.posthoc_nemenyi_friedman(
        value_table[METHOD_ORDER].to_numpy(dtype=float)
    )

    nemenyi_p = pd.DataFrame(
        np.asarray(nemenyi_result, dtype=float),
        index=METHOD_ORDER,
        columns=METHOD_ORDER,
    )

    nemenyi_p.to_csv(
        OUTPUT_DIR
        / f"nemenyi_pvalues_{metric}.csv"
    )

    significant = nemenyi_p < ALPHA
    sig_arr = significant.to_numpy(copy=True)
    np.fill_diagonal(sig_arr, False)

    significant = pd.DataFrame(
        sig_arr,
        index=METHOD_ORDER,
        columns=METHOD_ORDER,
    )

    significant.to_csv(
        OUTPUT_DIR
        / f"nemenyi_significant_{metric}.csv"
    )

    # Conventional CD reference value.
    cd, q_alpha = dem_sar_nemenyi_cd(
        k=k,
        n_datasets=n_datasets,
        alpha=ALPHA,
    )

    rank_long = (
        rank_table
        .reset_index()
        .melt(
            id_vars="dataset",
            var_name="method",
            value_name="rank",
        )
    )

    rank_long["metric"] = metric
    all_rank_rows.append(rank_long)

    friedman_rows.append({
        "metric": metric,
        "metric_label": info["label"],
        "direction": (
            "higher_is_better"
            if info["higher_is_better"]
            else "lower_is_better"
        ),
        "n_datasets": n_datasets,
        "n_methods": k,
        "friedman_statistic": float(friedman_stat),
        "friedman_p_value": float(friedman_p),
        "friedman_significant_0.05": bool(
            friedman_p < ALPHA
        ),
        "nemenyi_q_alpha": q_alpha,
        "nemenyi_cd": cd,
    })

    print(
        "Direction:",
        "higher is better"
        if info["higher_is_better"]
        else "lower is better",
    )

    print("Datasets (blocks):", n_datasets)
    print("Methods:", k)
    print(f"Friedman statistic: {friedman_stat:.6f}")
    print(f"Friedman p-value:   {friedman_p:.10g}")
    print(f"Nemenyi CD:         {cd:.6f}")

    print("\nAverage ranks (1 = best):")
    print(average_ranks.to_string())

    if friedman_p < ALPHA:
        print(
            "\nFriedman test is significant; "
            "Nemenyi post-hoc comparisons are interpreted."
        )
    else:
        print(
            "\nFriedman test is not significant at alpha=0.05; "
            "pairwise post-hoc claims should not be emphasized."
        )

    plot_cd_diagram(
        average_ranks=average_ranks,
        p_matrix=nemenyi_p,
        cd=cd,
        metric_label=info["label"],
        out_path=(
            OUTPUT_DIR
            / f"critical_diagram_{metric}.png"
        ),
        alpha=ALPHA,
        n_datasets=n_datasets,
    )


In [ ]:
# ============================================================
# SAVE GLOBAL STATISTICAL OUTPUTS
# ============================================================

friedman_summary = pd.DataFrame(
    friedman_rows
)

friedman_summary.to_csv(
    OUTPUT_DIR
    / "friedman_nemenyi_summary_all_metrics.csv",
    index=False,
)

all_ranks = pd.concat(
    all_rank_rows,
    ignore_index=True,
)

all_ranks.to_csv(
    OUTPUT_DIR
    / "all_dataset_method_metric_ranks.csv",
    index=False,
)

display_columns = [
    "metric_label",
    "direction",
    "n_datasets",
    "friedman_statistic",
    "friedman_p_value",
    "nemenyi_cd",
    "friedman_significant_0.05",
]

print(
    friedman_summary[
        display_columns
    ].to_string(index=False)
)


In [ ]:
# ============================================================
# DESCRIPTIVE OVERALL AVERAGE RANK ACROSS ALL METRICS
# ============================================================

overall_dataset_method = (
    all_ranks
    .groupby(
        ["dataset", "method"],
        as_index=False,
    )["rank"]
    .mean()
    .rename(
        columns={
            "rank": "mean_rank_across_metrics"
        }
    )
)

overall_average_ranks = (
    overall_dataset_method
    .groupby("method")[
        "mean_rank_across_metrics"
    ]
    .mean()
    .reindex(METHOD_ORDER)
    .sort_values()
)

overall_dataset_method.to_csv(
    OUTPUT_DIR
    / "overall_rank_by_dataset_method.csv",
    index=False,
)

overall_average_ranks.rename(
    "overall_average_rank"
).to_csv(
    OUTPUT_DIR
    / "overall_average_rank_across_metrics.csv"
)

print(
    "Overall descriptive average ranks:"
)
print(
    overall_average_ranks.to_string()
)


In [ ]:
# ============================================================
# BLACK OLD-STYLE OVERALL RANK PLOT
# ============================================================

def plot_overall_average_rank(
    overall_average_ranks,
    out_path,
    n_metrics,
    n_datasets=21,
):
    ordered = overall_average_ranks.sort_values()

    methods = ordered.index.tolist()
    ranks = ordered.to_numpy(dtype=float)
    k = len(methods)

    n_left = (k + 1) // 2
    left_idx = list(range(n_left))
    right_idx = list(range(n_left, k))

    fig, ax = plt.subplots(
        figsize=(12.8, 5.0)
    )

    axis_y = 0.0

    ax.hlines(
        axis_y,
        1,
        k,
        color="black",
        linewidth=1.3,
    )

    for r in np.arange(
        1.0,
        k + 0.001,
        0.5,
    ):
        major = abs(
            r - round(r)
        ) < 1e-10

        half_height = (
            0.075
            if major
            else 0.04
        )

        ax.vlines(
            r,
            axis_y - half_height,
            axis_y + half_height,
            color="black",
            linewidth=(
                1.2
                if major
                else 0.9
            ),
        )

    for r in range(1, k + 1):
        ax.text(
            r,
            0.17,
            str(r),
            ha="center",
            va="bottom",
            fontsize=11,
            color="black",
        )

    left_x = 0.72
    right_x = k + 0.28

    left_ys = np.linspace(
        -0.30,
        -1.28,
        len(left_idx),
    )

    right_ys = np.linspace(
        -0.30,
        -1.05,
        max(1, len(right_idx)),
    )

    for idx, y in zip(
        left_idx,
        left_ys,
    ):
        method = methods[idx]
        rank = ranks[idx]

        ax.plot(
            [rank, rank],
            [axis_y, y],
            color="black",
            linewidth=0.9,
        )

        ax.plot(
            [left_x, rank],
            [y, y],
            color="black",
            linewidth=0.9,
        )

        ax.text(
            left_x - 0.06,
            y,
            method,
            ha="right",
            va="center",
            fontsize=10.5,
            color="black",
        )

    for idx, y in zip(
        right_idx,
        right_ys,
    ):
        method = methods[idx]
        rank = ranks[idx]

        ax.plot(
            [rank, rank],
            [axis_y, y],
            color="black",
            linewidth=0.9,
        )

        ax.plot(
            [rank, right_x],
            [y, y],
            color="black",
            linewidth=0.9,
        )

        ax.text(
            right_x + 0.06,
            y,
            method,
            ha="left",
            va="center",
            fontsize=10.5,
            color="black",
        )

    ax.text(
        (1 + k) / 2,
        0.64,
        "Overall average rank across evaluation metrics",
        ha="center",
        va="bottom",
        fontsize=15,
        fontweight="bold",
        color="black",
    )

    ax.text(
        (1 + k) / 2,
        -1.62,
        (
            f"Descriptive summary over {n_metrics} "
            f"evaluation metrics and {n_datasets} datasets. "
            "Lower rank is better. All metrics receive equal weight; "
            "no Nemenyi significance claim is made for this "
            "composite ranking."
        ),
        ha="center",
        va="top",
        fontsize=9.3,
        color="black",
    )

    ax.set_xlim(
        0.48,
        k + 0.52,
    )

    ax.set_ylim(
        -1.87,
        0.97,
    )

    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.tight_layout()

    out_path = Path(out_path)

    fig.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        out_path.with_suffix(".pdf"),
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


plot_overall_average_rank(
    overall_average_ranks=(
        overall_average_ranks
    ),
    out_path=(
        OUTPUT_DIR
        / "overall_average_rank_across_metrics.png"
    ),
    n_metrics=len(METRICS),
    n_datasets=EXPECTED_DATASETS,
)


In [ ]:
# ============================================================
# FINAL SANITY CHECKS
# ============================================================

unique_cds = np.unique(
    np.round(
        friedman_summary[
            "nemenyi_cd"
        ].to_numpy(dtype=float),
        12,
    )
)

print(
    "Unique CD values across metrics:"
)
print(
    unique_cds
)

if len(unique_cds) != 1:
    raise AssertionError(
        "CD unexpectedly changed across metrics even though "
        "k, N, and alpha are fixed."
    )

direction_check = pd.DataFrame(
    [
        {
            "metric": metric,
            "paper_label": info["label"],
            "better_direction": (
                "higher"
                if info[
                    "higher_is_better"
                ]
                else "lower"
            ),
        }
        for metric, info in METRICS.items()
    ]
)

direction_check.to_csv(
    OUTPUT_DIR
    / "metric_rank_directions.csv",
    index=False,
)

print(
    "\nMetric ranking directions:"
)
print(
    direction_check.to_string(
        index=False
    )
)

print(
    "\nAll corrected outputs saved to:"
)
print(
    OUTPUT_DIR.resolve()
)

print(
    "\nInterpretation:\n"
    "  * Per-metric Friedman/Nemenyi tests use 21 datasets as blocks.\n"
    "  * rank 1 is best.\n"
    "  * distance/proximity/sparsity/runtime are lower-is-better.\n"
    "  * crossbars are derived from the scikit-posthocs Nemenyi p-value matrix.\n"
    "  * crossbars mean no significant difference detected at alpha=0.05.\n"
    "  * the overall average-rank plot is descriptive only."
)


# Dataset-level Runtime analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ============================================================
# Dataset-macro runtime summary
# ============================================================

runtime_summary = (
    results
    .groupby("method", as_index=False)
    .agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        std_runtime_sec=("runtime_sec", "std"),
        median_runtime_sec=("runtime_sec", "median"),
        min_runtime_sec=("runtime_sec", "min"),
        max_runtime_sec=("runtime_sec", "max"),
        n_datasets=("dataset", "nunique"),
    )
)

# Standard error and 95% t confidence interval of the
# dataset-level macro mean.
runtime_summary["sem_runtime_sec"] = (
    runtime_summary["std_runtime_sec"]
    / np.sqrt(runtime_summary["n_datasets"])
)

runtime_summary["t_crit_95"] = runtime_summary[
    "n_datasets"
].apply(
    lambda n: t.ppf(0.975, df=n - 1)
)

runtime_summary["ci95_halfwidth"] = (
    runtime_summary["t_crit_95"]
    * runtime_summary["sem_runtime_sec"]
)

runtime_summary["ci95_low"] = (
    runtime_summary["mean_runtime_sec"]
    - runtime_summary["ci95_halfwidth"]
)

runtime_summary["ci95_high"] = (
    runtime_summary["mean_runtime_sec"]
    + runtime_summary["ci95_halfwidth"]
)

# Safety check for log-scale plotting
if (runtime_summary["ci95_low"] <= 0).any():
    raise ValueError(
        "At least one 95% CI reaches zero or below; "
        "use a bootstrap/log-scale interval instead."
    )

runtime_summary = (
    runtime_summary
    .sort_values("mean_runtime_sec")
    .reset_index(drop=True)
)

print(runtime_summary)

runtime_summary.to_csv(
    OUTPUT_DIR / "runtime_summary_all_21_datasets.csv",
    index=False
)


# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(8.5, 4.8))

plot_df = runtime_summary.copy()

x = np.arange(len(plot_df))

# Asymmetric error bars from CI bounds
lower_err = (
    plot_df["mean_runtime_sec"]
    - plot_df["ci95_low"]
)

upper_err = (
    plot_df["ci95_high"]
    - plot_df["mean_runtime_sec"]
)

yerr = np.vstack([
    lower_err.to_numpy(),
    upper_err.to_numpy(),
])

ax.bar(
    x,
    plot_df["mean_runtime_sec"],
    yerr=yerr,
    capsize=4,
    edgecolor="black",
    facecolor="white",
    linewidth=1.2,
)

# Optional: show the actual 21 dataset-level runtime means
# behind each method
for i, method in enumerate(plot_df["method"]):
    vals = (
        results.loc[
            results["method"] == method,
            "runtime_sec"
        ]
        .to_numpy()
    )

    # Small deterministic horizontal spread so points do not overlap
    offsets = np.linspace(-0.10, 0.10, len(vals))

    ax.scatter(
        np.full(len(vals), i) + offsets,
        vals,
        s=10,
        alpha=0.45,
        zorder=3,
    )

ax.set_yscale("log")

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["method"],
    rotation=35,
    ha="right",
)

ax.set_ylabel(
    "Runtime per counterfactual call (s, log scale)"
)

ax.set_title(
    "Counterfactual generation runtime across 21 datasets",
    fontweight="bold",
)

ax.grid(
    axis="y",
    which="both",
    linestyle="--",
    alpha=0.30,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "runtime_all_21_datasets.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_DIR / "runtime_all_21_datasets.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ============================================================
# Average runtime across all 21 datasets
# ============================================================

# results already contains:
# 17 BRGM datasets + 4 public datasets
# one row per (dataset, method)

runtime_by_method = (
    results
    .groupby("method", as_index=False)
    .agg(
        mean_runtime_sec=("runtime_sec", "mean"),
        std_runtime_sec=("runtime_sec", "std"),
        median_runtime_sec=("runtime_sec", "median"),
        min_runtime_sec=("runtime_sec", "min"),
        max_runtime_sec=("runtime_sec", "max"),
        n_datasets=("dataset", "nunique"),
    )
    .sort_values("mean_runtime_sec")
    .reset_index(drop=True)
)

print(runtime_by_method)

# Safety check
assert (runtime_by_method["n_datasets"] == 21).all(), \
    "Each method should have runtime results for all 21 datasets."

runtime_by_method.to_csv(
    OUTPUT_DIR / "runtime_summary_all_21_datasets.csv",
    index=False
)

In [ ]:
# ============================================================
# Runtime figure across all 21 datasets
# ============================================================

fig, ax = plt.subplots(figsize=(8, 4.8))

plot_df = runtime_by_method.copy()

x = np.arange(len(plot_df))

ax.bar(
    x,
    plot_df["mean_runtime_sec"],
    yerr=plot_df["std_runtime_sec"],
    capsize=4,
    edgecolor="black",
    facecolor="white",
    linewidth=1.2,
)

ax.set_yscale("log")

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["method"],
    rotation=35,
    ha="right",
)

ax.set_ylabel("Runtime per counterfactual call (s, log scale)")

ax.set_title(
    "Counterfactual generation runtime across 21 datasets",
    fontweight="bold",
)

ax.grid(
    axis="y",
    which="both",
    linestyle="--",
    alpha=0.35,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "runtime_all_21_datasets.png",
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    OUTPUT_DIR / "runtime_all_21_datasets.pdf",
    bbox_inches="tight",
)

plt.show()

# Best Forecaster

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Full-test R² of the selected GRU forecaster for all datasets
# ============================================================

results = pd.DataFrame({
    "dataset": [
        "piezo1", "piezo2", "piezo3", "piezo4", "piezo5",
        "piezo6", "piezo7", "piezo8", "piezo9", "piezo11",
        "piezo12", "piezo13", "piezo14", "piezo15", "piezo16",
        "piezo17", "piezo18",
        "ETTh1", "ETTh2", "Weather", "Electricity"
    ],

    "test_r2": [
        0.405,
        0.999,
        0.941,
        0.983,
        0.938,
        0.994,
        0.978,
        0.920,
        1.000,
        0.999,
        0.997,
        0.975,
        0.836,
        0.969,
        0.643,
        0.761,
        0.861,
        0.5161,
        0.8103,
        0.9708,
        0.7938
    ],

    "group": (
        ["BRGM"] * 17 +
        ["Public"] * 4
    )
})

# ============================================================
# Plot
# ============================================================

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(results))

brgm = results["group"] == "BRGM"
public = results["group"] == "Public"

ax.bar(
    x[brgm],
    results.loc[brgm, "test_r2"],
    label="BRGM"
)

ax.bar(
    x[public],
    results.loc[public, "test_r2"],
    label="Public datasets"
)

# Separator between BRGM and public datasets
ax.axvline(16.5, linestyle="--", linewidth=1)

ax.set_ylabel(r"Full-test $R^2$")
ax.set_xlabel("Dataset")

ax.set_xticks(x)
ax.set_xticklabels(
    results["dataset"],
    rotation=45,
    ha="right"
)

# Give extra vertical room for value labels
ax.set_ylim(0, 1.10)

ax.grid(
    axis="y",
    alpha=0.25
)

# ============================================================
# Add R² value above EVERY bar
# ============================================================

for i, value in enumerate(results["test_r2"]):
    ax.text(
        i,
        value + 0.015,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=8
    )

# ============================================================
# Legend outside the plotting area
# ============================================================

ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    frameon=False
)

fig.tight_layout(rect=[0, 0, 0.88, 1])

# Save
fig.savefig(
    OUTPUT_DIR / "gru_full_test_r2_21_datasets.pdf",
    bbox_inches="tight"
)

fig.savefig(
    OUTPUT_DIR / "gru_full_test_r2_21_datasets.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Per-horizon-step Validity Ratio
for the public datasets

In [ ]:
# ============================================================
# Per-horizon validity plots for EACH public dataset separately
# ============================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
RESULTS_DIR = PROJECT_ROOT / "Results" / "EDBT-results-public-datasets"
PLOT_DIR = RESULTS_DIR / "plots" / "per_dataset_horizon_validity"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("Results directory:", RESULTS_DIR)
print("Plot directory:", PLOT_DIR)


In [ ]:
PUBLIC_DATASETS = [
    "ETTh1",
    "ETTh2",
    "Weather",
    "Electricity",
]

# Keep exactly the same method ordering used in the benchmark.
METHODS_TO_PLOT = METHOD_ORDER


In [ ]:
# ------------------------------------------------------------
# LOAD ONE DATASET'S RAW RESULTS
# ------------------------------------------------------------

def load_public_dataset_raw_results(
    results_dir: Path,
    dataset_name: str,
) -> pd.DataFrame:

    csv_path = (
        results_dir
        / "datasets"
        / dataset_name
        / "counterfactual_results.csv"
    )

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Could not find results for {dataset_name}:\n"
            f"{csv_path}"
        )

    raw = pd.read_csv(csv_path)

    # Same filtering used by your original plotting function.
    if "status" in raw.columns:
        raw = raw[
            raw["status"].astype(str) == "ok"
        ].copy()

    return raw

In [ ]:
# ------------------------------------------------------------
# COMPUTE PER-HORIZON VALIDITY FOR ONE DATASET
# ------------------------------------------------------------

def compute_per_horizon_validity_for_dataset(
    raw: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:

    valid_cols = sorted(
        [
            c for c in raw.columns
            if re.fullmatch(r"valid_h_\d+", c)
        ],
        key=lambda c: int(c.split("_")[-1]),
    )

    if not valid_cols:
        raise ValueError(
            f"No valid_h_XX columns found for {dataset_name}.\n"
            "The saved counterfactual_results.csv must contain "
            "valid_h_01 ... valid_h_24."
        )

    horizon_steps = np.arange(
        1,
        len(valid_cols) + 1,
    )

    rows = []

    for method in METHODS_TO_PLOT:

        method_rows = raw[
            raw["method"] == method
        ].copy()

        if len(method_rows) == 0:
            print(
                f"Warning: no rows for {method} "
                f"in {dataset_name}"
            )
            continue

        step_values = (
            method_rows[valid_cols]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
        )

        # Each valid_h_XX column contains 0/1.
        # Therefore its mean is exactly the proportion
        # of counterfactuals valid at that horizon step.
        per_step_ratio = (
            step_values
            .mean(axis=0)
            .to_numpy(dtype=float)
        )

        for h, value in zip(
            horizon_steps,
            per_step_ratio,
        ):
            rows.append({
                "dataset": dataset_name,
                "method": method,
                "horizon_step": int(h),
                "validity_ratio_at_step": float(value),
                "n_examples": int(len(method_rows)),
            })

    return pd.DataFrame(rows)

In [ ]:
# ------------------------------------------------------------
# PLOT ONE DATASET
# Same style as the original BRGM/public per-horizon figure
# ------------------------------------------------------------

def plot_per_horizon_validity_one_dataset(
    results_dir: Path,
    dataset_name: str,
) -> pd.DataFrame:

    raw = load_public_dataset_raw_results(
        results_dir=results_dir,
        dataset_name=dataset_name,
    )
    horizon_df = (
        compute_per_horizon_validity_for_dataset(
            raw=raw,
            dataset_name=dataset_name,
        )
    )

    dataset_plot_dir = (
        results_dir
        / "plots"
        / "per_dataset_horizon_validity"
    )

    dataset_plot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # Save exact values used in the figure
    csv_out = (
        dataset_plot_dir
        / f"{dataset_name}_per_horizon_validity.csv"
    )

    horizon_df.to_csv(
        csv_out,
        index=False,
    )

    horizon_steps = sorted(
        horizon_df["horizon_step"].unique()
    )

    # ---- plotting ----
    plt.figure(
        figsize=(11, 6)
    )

    for method in METHODS_TO_PLOT:

        method_df = horizon_df[
            horizon_df["method"] == method
        ]

        if len(method_df) == 0:
            continue

        plt.plot(
            method_df["horizon_step"],
            method_df["validity_ratio_at_step"],
            marker="o",
            linewidth=2,
            label=method,
        )

    plt.xlabel(
        "Forecast horizon step"
    )

    plt.ylabel(
        "Validity ratio at horizon step"
    )

    plt.title(
        f"Per-horizon-step validity ratio by method — {dataset_name}"
    )

    plt.ylim(
        -0.05,
        1.05,
    )

    plt.xticks(
        horizon_steps
    )

    plt.grid(
        True,
        alpha=0.25,
    )

    plt.legend(
        fontsize=8
    )

    plt.tight_layout()

    png_out = (
        dataset_plot_dir
        / f"{dataset_name}_per_horizon_validity.png"
    )

    pdf_out = (
        dataset_plot_dir
        / f"{dataset_name}_per_horizon_validity.pdf"
    )

    plt.savefig(
        png_out,
        dpi=300,
        bbox_inches="tight",
    )

    plt.savefig(
        pdf_out,
        bbox_inches="tight",
    )

    plt.show()

    print(
        f"\nSaved {dataset_name}:"
    )
    print("Plot:", png_out)
    print("CSV: ", csv_out)

    return horizon_df

In [ ]:
# ------------------------------------------------------------
# RUN FOR ALL FOUR PUBLIC DATASETS
# ------------------------------------------------------------

all_public_horizon_results = []

for dataset_name in PUBLIC_DATASETS:

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)

    df_dataset = (
        plot_per_horizon_validity_one_dataset(
            results_dir=RESULTS_DIR,
            dataset_name=dataset_name,
        )
    )

    all_public_horizon_results.append(
        df_dataset
    )


In [ ]:
# ------------------------------------------------------------
# OPTIONAL: save one combined CSV containing all four datasets
# ------------------------------------------------------------

public_horizon_all = pd.concat(
    all_public_horizon_results,
    ignore_index=True,
)

combined_csv = (
    RESULTS_DIR
    / "plots"
    / "per_dataset_horizon_validity"
    / "all_public_datasets_per_horizon_validity.csv"
)

public_horizon_all.to_csv(
    combined_csv,
    index=False,
)

print(
    "\nCombined CSV saved to:"
)
print(combined_csv)

# Dataset Independence Analysis

In [ ]:
# ============================================================
# DATASET DEPENDENCE / SIMILARITY ANALYSIS
#
# Goal:
#   1. Compare the 17 BRGM piezometric target series
#   2. Compare ETTh1 and ETTh2
#
# Analyses:
#   - Spearman correlation of target levels
#   - Spearman correlation of first differences
#   - Maximum lagged Spearman correlation
#   - Circular-shift permutation test for pairwise dependence
#
# IMPORTANT:
# Correlation != statistical independence.
# This analysis detects strong shared temporal/monotonic structure.
# ============================================================

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

In [ ]:
# ============================================================
# 1. PATHS
# ============================================================

PIEZO_DIR = PROJECT_ROOT / "Data" / "piezos"
ETTH1_FILE = PROJECT_ROOT / "Data" / "Other" / "ETT-small" / "ETTh1.csv"
ETTH2_FILE = PROJECT_ROOT / "Data" / "Other" / "ETT-small" / "ETTh2.csv"
OUTPUT_DIR = PROJECT_ROOT / "Results" / "nemenyi_results_corrected" / "dataset_dependence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

PIEZO_TARGET = "level"
ETT_TARGET = "OT"

# Candidate timestamp-column names.
TIME_CANDIDATES = [
    "date",
    "datetime",
    "timestamp",
    "time",
    "Date",
    "Datetime",
    "Timestamp",
]


# Maximum lag:
# piezometers -> ±30 days
PIEZO_MAX_LAG = 30

# ETT -> ±48 hours
ETT_MAX_LAG = 48

# Permutations for the dependence test.
# 1000 is sufficient for exploratory analysis.
N_PERMUTATIONS = 1000

RANDOM_SEED = 42

In [ ]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def find_time_column(df):
    """
    Find a likely timestamp column.
    """
    for col in TIME_CANDIDATES:
        if col in df.columns:
            return col

    raise ValueError(
        f"No timestamp column found. Columns are: {list(df.columns)}"
    )


def load_target_series(file_path, target_column):
    """
    Read one dataset and return its target as a pandas Series
    indexed by datetime.
    """

    df = pd.read_csv(file_path)

    time_col = find_time_column(df)

    if target_column not in df.columns:
        raise ValueError(
            f"{file_path.name}: target '{target_column}' not found. "
            f"Columns = {list(df.columns)}"
        )

    df[time_col] = pd.to_datetime(
        df[time_col],
        errors="coerce"
    )

    df[target_column] = pd.to_numeric(
        df[target_column],
        errors="coerce"
    )

    df = (
        df[[time_col, target_column]]
        .dropna()
        .drop_duplicates(subset=time_col)
        .sort_values(time_col)
    )

    series = df.set_index(time_col)[target_column]

    return series


def align_two_series(s1, s2):
    """
    Inner join by actual timestamp.

    This is essential: correlation should only be computed
    where both datasets have observations at the same time.
    """

    aligned = pd.concat(
        [s1, s2],
        axis=1,
        join="inner"
    ).dropna()

    aligned.columns = ["x", "y"]

    return aligned


def safe_spearman(x, y):
    """
    Spearman rho with basic safety checks.
    """

    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)

    x = x[mask]
    y = y[mask]

    if len(x) < 10:
        return np.nan, np.nan

    if np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan, np.nan

    result = spearmanr(x, y)

    return result.statistic, result.pvalue

In [ ]:
# ============================================================
# 4. LOAD THE 17 PIEZOMETER SERIES
# ============================================================

# Adjust the filename pattern if necessary.
piezo_files = sorted(
    PIEZO_DIR.glob("*.csv")
)
print("CSV files found:")
for f in piezo_files:
    print(f.name)


piezo_series = {}

for file in piezo_files:

    # expected filenames such as piezo1.csv, piezo2.csv, ...
    name = file.stem

    try:
        s = load_target_series(
            file,
            PIEZO_TARGET
        )

        piezo_series[name] = s

    except Exception as e:
        print(
            f"Skipping {file.name}: {e}"
        )


print(
    "\nPiezometer datasets loaded:",
    len(piezo_series)
)

if len(piezo_series) != 17:
    print(
        "WARNING: expected 17 piezometers. "
        "Check PIEZO_DIR and filename selection."
    )


In [ ]:
# ============================================================
# 5. PAIRWISE SPEARMAN CORRELATION — RAW TARGET LEVELS
# ============================================================

names = list(piezo_series.keys())

level_corr = pd.DataFrame(
    np.nan,
    index=names,
    columns=names
)

overlap_count = pd.DataFrame(
    0,
    index=names,
    columns=names
)


for name1 in names:

    for name2 in names:

        if name1 == name2:

            level_corr.loc[name1, name2] = 1.0
            overlap_count.loc[name1, name2] = len(
                piezo_series[name1]
            )

            continue

        aligned = align_two_series(
            piezo_series[name1],
            piezo_series[name2]
        )

        overlap_count.loc[name1, name2] = len(aligned)

        if len(aligned) >= 10:

            rho, _ = safe_spearman(
                aligned["x"],
                aligned["y"]
            )

            level_corr.loc[
                name1,
                name2
            ] = rho


level_corr.to_csv(
    OUTPUT_DIR /
    "piezo_spearman_target_levels.csv"
)

overlap_count.to_csv(
    OUTPUT_DIR /
    "piezo_pairwise_timestamp_overlap.csv"
)

In [ ]:
# ============================================================
# 6. HEATMAP — RAW LEVELS
# ============================================================

fig, ax = plt.subplots(
    figsize=(11, 9)
)

im = ax.imshow(
    level_corr.astype(float).values,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    np.arange(len(names))
)

ax.set_yticks(
    np.arange(len(names))
)

ax.set_xticklabels(
    names,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    names
)

for i in range(len(names)):
    for j in range(len(names)):

        value = level_corr.iloc[i, j]

        if pd.notna(value):

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=7
            )


cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Spearman $\\rho$"
)

ax.set_title(
    "Spearman correlation between piezometer target levels"
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR /
    "piezo_spearman_levels_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 7. FIRST DIFFERENCES
#
# This is VERY important.
#
# Instead of:
#       level(t)
#
# compare:
#       level(t) - level(t-1)
#
# This reduces correlations caused only by common trend /
# seasonality and asks whether the actual changes co-move.
# ============================================================

diff_series = {}

for name, series in piezo_series.items():

    diff_series[name] = (
        series
        .diff()
        .dropna()
    )


diff_corr = pd.DataFrame(
    np.nan,
    index=names,
    columns=names
)


for name1 in names:

    for name2 in names:

        if name1 == name2:

            diff_corr.loc[
                name1,
                name2
            ] = 1.0

            continue

        aligned = align_two_series(
            diff_series[name1],
            diff_series[name2]
        )

        if len(aligned) >= 10:

            rho, _ = safe_spearman(
                aligned["x"],
                aligned["y"]
            )

            diff_corr.loc[
                name1,
                name2
            ] = rho
            diff_corr.to_csv(
    OUTPUT_DIR /
    "piezo_spearman_first_differences.csv"
)

In [ ]:
# ============================================================
# 8. HEATMAP — FIRST DIFFERENCES
# ============================================================

fig, ax = plt.subplots(
    figsize=(11, 9)
)

im = ax.imshow(
    diff_corr.astype(float).values,
    vmin=-1,
    vmax=1
)

ax.set_xticks(
    np.arange(len(names))
)

ax.set_yticks(
    np.arange(len(names))
)

ax.set_xticklabels(
    names,
    rotation=45,
    ha="right"
)

ax.set_yticklabels(
    names
)

for i in range(len(names)):
    for j in range(len(names)):

        value = diff_corr.iloc[i, j]

        if pd.notna(value):

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=7
            )


cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Spearman $\\rho$"
)

ax.set_title(
    "Spearman correlation between daily changes "
    "in piezometer target levels"
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR /
    "piezo_spearman_first_difference_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 9. MAXIMUM LAGGED SPEARMAN CORRELATION
# ============================================================

def maximum_lagged_spearman(
    s1,
    s2,
    max_lag=30
):
    """
    Finds the maximum absolute Spearman correlation
    over lags [-max_lag, ..., +max_lag].

    Positive lag means the second series is shifted.
    """

    aligned = align_two_series(
        s1,
        s2
    )

    if len(aligned) < 20:
        return np.nan, np.nan

    best_rho = np.nan
    best_lag = np.nan

    x = aligned["x"]
    y = aligned["y"]

    for lag in range(
        -max_lag,
        max_lag + 1
    ):

        y_shifted = y.shift(lag)

        pair = pd.concat(
            [x, y_shifted],
            axis=1
        ).dropna()

        if len(pair) < 10:
            continue

        rho, _ = safe_spearman(
            pair.iloc[:, 0],
            pair.iloc[:, 1]
        )

        if np.isnan(rho):
            continue

        if (
            np.isnan(best_rho)
            or abs(rho) > abs(best_rho)
        ):

            best_rho = rho
            best_lag = lag

    return best_rho, best_lag


lag_rows = []

for name1, name2 in combinations(
    names,
    2
):

    rho, lag = maximum_lagged_spearman(
        diff_series[name1],
        diff_series[name2],
        max_lag=PIEZO_MAX_LAG
    )

    lag_rows.append({
        "dataset_1": name1,
        "dataset_2": name2,
        "max_abs_lagged_spearman": rho,
        "best_lag_days": lag,
    })


lag_results = pd.DataFrame(
    lag_rows
)

lag_results[
    "abs_max_lagged_spearman"
] = (
    lag_results[
        "max_abs_lagged_spearman"
    ].abs()
)


lag_results = lag_results.sort_values(
    "abs_max_lagged_spearman",
    ascending=False
)


lag_results.to_csv(
    OUTPUT_DIR /
    "piezo_max_lagged_spearman.csv",
    index=False
)

print(
    "\nStrongest lagged relationships:"
)

print(
    lag_results.head(20)
    .round(3)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# 10. CIRCULAR-SHIFT PERMUTATION TEST
#
# Ordinary Spearman p-values assume independent observations.
# Time-series observations are autocorrelated.
#
# A circular shift preserves the internal time structure of
# each series while breaking their alignment with each other.
# ============================================================

def circular_shift_test(
    s1,
    s2,
    n_permutations=1000,
    seed=42
):
    """
    Test whether |Spearman rho| is stronger than expected if
    the two time series were temporally misaligned.

    Applied to FIRST DIFFERENCES.
    """

    aligned = align_two_series(
        s1,
        s2
    )

    x = aligned["x"].to_numpy()
    y = aligned["y"].to_numpy()

    if len(x) < 20:
        return np.nan, np.nan, np.nan
    observed_rho, _ = safe_spearman(
        x,
        y
    )

    rng = np.random.default_rng(seed)

    permutation_rhos = []

    n = len(y)

    # Avoid extremely tiny shifts.
    minimum_shift = max(
        5,
        int(0.05 * n)
    )

    maximum_shift = n - minimum_shift

    if maximum_shift <= minimum_shift:
        return observed_rho, np.nan, np.nan

    for _ in range(
        n_permutations
    ):

        shift = rng.integers(
            minimum_shift,
            maximum_shift
        )

        y_shift = np.roll(
            y,
            shift
        )

        rho, _ = safe_spearman(
            x,
            y_shift
        )

        permutation_rhos.append(
            rho
        )

    permutation_rhos = np.asarray(
        permutation_rhos
    )

    # Two-sided empirical p-value.
    p_value = (
        1
        + np.sum(
            np.abs(permutation_rhos)
            >= abs(observed_rho)
        )
    ) / (
        n_permutations + 1
    )

    return (
        observed_rho,
        p_value,
        len(x)
    )


permutation_rows = []

for pair_index, (
    name1,
    name2
) in enumerate(
    combinations(names, 2)
):

    rho, p_value, n_overlap = (
        circular_shift_test(
            diff_series[name1],
            diff_series[name2],
            n_permutations=N_PERMUTATIONS,
            seed=RANDOM_SEED + pair_index
        )
    )

    permutation_rows.append({
        "dataset_1": name1,
        "dataset_2": name2,
        "spearman_diff": rho,
        "permutation_p": p_value,
        "n_overlap": n_overlap,
    })


permutation_results = pd.DataFrame(
    permutation_rows
)

In [ ]:
# ============================================================
# 11. BENJAMINI-HOCHBERG CORRECTION
# ============================================================

def benjamini_hochberg(p_values):

    p = np.asarray(
        p_values,
        dtype=float
    )

    valid = np.isfinite(p)

    adjusted = np.full(
        len(p),
        np.nan
    )

    p_valid = p[valid]

    m = len(p_valid)

    order = np.argsort(
        p_valid
    )

    ranked = p_valid[order]

    q_ranked = (
        ranked
        * m
        / np.arange(
            1,
            m + 1
        )
    )

    q_ranked = np.minimum.accumulate(
        q_ranked[::-1]
    )[::-1]

    q_ranked = np.clip(
        q_ranked,
        0,
        1
    )

    q = np.empty(
        m
    )

    q[order] = q_ranked

    adjusted[valid] = q

    return adjusted


permutation_results[
    "p_fdr_bh"
] = benjamini_hochberg(
    permutation_results[
        "permutation_p"
    ]
)


permutation_results[
    "significant_fdr_0.05"
] = (
    permutation_results[
        "p_fdr_bh"
    ] < 0.05
)


permutation_results[
    "abs_spearman_diff"
] = (
    permutation_results[
        "spearman_diff"
    ].abs()
)


permutation_results = (
    permutation_results
    .sort_values(
        "abs_spearman_diff",
        ascending=False
    )
)


permutation_results.to_csv(
    OUTPUT_DIR /
    "piezo_dependence_circular_shift_test.csv",
    index=False
)


print(
    "\nPairwise dependence results "
    "(ranked by first-difference correlation):"
)

print(
    permutation_results[
        [
            "dataset_1",
            "dataset_2",
            "spearman_diff",
            "permutation_p",
            "p_fdr_bh",
            "significant_fdr_0.05",
            "n_overlap",
        ]
    ]
    .head(30)
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# 12. SUMMARY OF HIGH-CORRELATION PIEZOMETER PAIRS
# ============================================================

summary_rows = []

for name1, name2 in combinations(
    names,
    2
):

    level_rho = level_corr.loc[
        name1,
        name2
    ]

    diff_rho = diff_corr.loc[
        name1,
        name2
    ]
    lag_row = lag_results[
        (
            (lag_results["dataset_1"] == name1)
            &
            (lag_results["dataset_2"] == name2)
        )
        |
        (
            (lag_results["dataset_1"] == name2)
            &
            (lag_results["dataset_2"] == name1)
        )
    ].iloc[0]

    perm_row = permutation_results[
        (
            (permutation_results["dataset_1"] == name1)
            &
            (permutation_results["dataset_2"] == name2)
        )
        |
        (
            (permutation_results["dataset_1"] == name2)
            &
            (permutation_results["dataset_2"] == name1)
        )
    ].iloc[0]

    summary_rows.append({
        "dataset_1": name1,
        "dataset_2": name2,

        "rho_levels": level_rho,
        "rho_first_difference": diff_rho,

        "max_lagged_rho":
            lag_row[
                "max_abs_lagged_spearman"
            ],

        "best_lag":
            lag_row[
                "best_lag_days"
            ],

        "permutation_p":
            perm_row[
                "permutation_p"
            ],

        "fdr_q":
            perm_row[
                "p_fdr_bh"
            ],

        "significant_fdr":
            perm_row[
                "significant_fdr_0.05"
            ],
    })


pair_summary = pd.DataFrame(
    summary_rows
)

pair_summary[
    "abs_diff_rho"
] = (
    pair_summary[
        "rho_first_difference"
    ].abs()
)


pair_summary = pair_summary.sort_values(
    "abs_diff_rho",
    ascending=False
)


pair_summary.to_csv(
    OUTPUT_DIR /
    "piezo_pairwise_summary.csv",
    index=False
)


print(
    "\nTop piezometer similarities:"
)

print(
    pair_summary.head(20)
    .round(3)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# 13. ETTh1 VS ETTh2
# ============================================================

etth1 = load_target_series(
    ETTH1_FILE,
    ETT_TARGET
)

etth2 = load_target_series(
    ETTH2_FILE,
    ETT_TARGET
)


ett_aligned = align_two_series(
    etth1,
    etth2
)


print(
    "\nETTh1 / ETTh2 aligned observations:",
    len(ett_aligned)
)


# ------------------------------------------------------------
# Raw OT levels
# ------------------------------------------------------------

ett_level_rho, ett_level_p = (
    safe_spearman(
        ett_aligned["x"],
        ett_aligned["y"]
    )
)


# ------------------------------------------------------------
# First differences
# ------------------------------------------------------------

ett_diff1 = etth1.diff().dropna()
ett_diff2 = etth2.diff().dropna()

ett_diff_aligned = align_two_series(
    ett_diff1,
    ett_diff2
)

ett_diff_rho, ett_diff_standard_p = (
    safe_spearman(
        ett_diff_aligned["x"],
        ett_diff_aligned["y"]
    )
)


# ------------------------------------------------------------
# Circular-shift test
# ------------------------------------------------------------

ett_perm_rho, ett_perm_p, ett_n = (
    circular_shift_test(
        ett_diff1,
        ett_diff2,
        n_permutations=N_PERMUTATIONS,
        seed=RANDOM_SEED
    )
)


# ------------------------------------------------------------
# Maximum lagged dependence
# ------------------------------------------------------------

ett_lag_rho, ett_best_lag = (
    maximum_lagged_spearman(
        ett_diff1,
        ett_diff2,
        max_lag=ETT_MAX_LAG
    )
)


ett_summary = pd.DataFrame(
    [{
        "dataset_1": "ETTh1",
        "dataset_2": "ETTh2",

        "n_aligned_levels":
            len(ett_aligned),

        "spearman_levels":
            ett_level_rho,

        "spearman_levels_standard_p":
            ett_level_p,

        "n_aligned_first_differences":
            len(ett_diff_aligned),

        "spearman_first_difference":
            ett_diff_rho,

        "first_difference_standard_p":
            ett_diff_standard_p,

        "circular_shift_permutation_p":
            ett_perm_p,

        "max_lagged_spearman":
            ett_lag_rho,
            "best_lag_hours":
            ett_best_lag,
    }]
)


ett_summary.to_csv(
    OUTPUT_DIR /
    "ETTh1_ETTh2_dependence_summary.csv",
    index=False
)


print(
    "\nETTh1 vs ETTh2:"
)

print(
    ett_summary
    .round(4)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# 14. ETTh1 / ETTh2 VISUAL COMPARISON
# ============================================================

# Standardize only for plotting so the shapes can be compared.
plot_data = ett_aligned.copy()

plot_data["ETTh1"] = (
    (
        plot_data["x"]
        - plot_data["x"].mean()
    )
    / plot_data["x"].std()
)

plot_data["ETTh2"] = (
    (
        plot_data["y"]
        - plot_data["y"].mean()
    )
    / plot_data["y"].std()
)


fig, ax = plt.subplots(
    figsize=(12, 5)
)

ax.plot(
    plot_data.index,
    plot_data["ETTh1"],
    label="ETTh1"
)

ax.plot(
    plot_data.index,
    plot_data["ETTh2"],
    label="ETTh2",
    alpha=0.75
)

ax.set_xlabel(
    "Time"
)

ax.set_ylabel(
    "Standardized OT"
)

ax.set_title(
    "ETTh1 and ETTh2 target trajectories"
)

ax.legend()

ax.grid(
    True,
    alpha=0.25
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR /
    "ETTh1_ETTh2_target_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# 15. FINAL FILES
# ============================================================

print("\nDone.")
print("Results saved to:")
print(OUTPUT_DIR)

print("\nPlease send me these files:")

print(
    "1. piezo_spearman_levels_heatmap.png"
)

print(
    "2. piezo_spearman_first_difference_heatmap.png"
)

print(
    "3. piezo_pairwise_summary.csv"
)

print(
    "4. piezo_dependence_circular_shift_test.csv"
)

print(
    "5. ETTh1_ETTh2_dependence_summary.csv"
)

print(
    "6. ETTh1_ETTh2_target_comparison.png"
)
